# Payment Gateway Processing Simulation

A prototype that models a unified payment-processing service across **Paytm** and **Razorpay**. It demonstrates requirements-driven validation, provider routing, retry handling, payment status transitions, audit history, and structured test scenarios.

## Business objective

Create a common payment interface that can:

- route a request to the selected payment provider;
- validate provider-specific business rules;
- initiate and confirm a transaction;
- retry transient failures without retrying invalid requests;
- store transaction results and status history for troubleshooting.

## Simplified payment flow

```text
Payment request
      │
      ▼
Gateway Factory ──► Paytm / Razorpay
      │
      ▼
Validation ──► Initiation ──► Confirmation
      │              │               │
      └──── failure / retry handling ┘
                     
```

> **Scope:** This notebook is an educational simulation. It does not call live payment APIs or process real funds.


In [1]:
import random
from dataclasses import dataclass
from enum import Enum, auto
from abc import ABC, abstractmethod

In [3]:
@dataclass
class PaymentRequest:
    sender: str
    receiver: str
    amount: float
    currency: str

In [5]:
class BankingSystem(ABC):
    @abstractmethod
    def process_payment(self, amount: float) -> bool:
        raise NotImplementedError


class PaytmBankingSystem(BankingSystem):
    def __init__(self) -> None:
        self._rand = random.Random()

    def process_payment(self, amount: float) -> bool:
        # Simulate 80% success (matches your Java behavior)
        r = self._rand.randrange(100)
        return r < 80


class RazorpayBankingSystem(BankingSystem):
    def __init__(self) -> None:
        self._rand = random.Random()

    def process_payment(self, amount: float) -> bool:
        print(f"[BankingSystem-Razorpay] Processing payment of {amount}...")
        # Simulate 90% success
        r = self._rand.randrange(100)
        return r < 90

class PaymentGateway(ABC):
    def __init__(self) -> None:
        self.banking_system: BankingSystem | None = None

    # Template method defining the standard payment flow
    def process_payment(self, request: PaymentRequest) -> bool:
        if not self.validate_payment(request):
            print(f"[PaymentGateway] Validation failed for {request.sender}.")
            return False
        if not self.initiate_payment(request):
            print(f"[PaymentGateway] Initiation failed for {request.sender}.")
            return False
        if not self.confirm_payment(request):
            print(f"[PaymentGateway] Confirmation failed for {request.sender}.")
            return False
        return True

    @abstractmethod
    def validate_payment(self, request: PaymentRequest) -> bool:
        raise NotImplementedError

    @abstractmethod
    def initiate_payment(self, request: PaymentRequest) -> bool:
        raise NotImplementedError

    @abstractmethod
    def confirm_payment(self, request: PaymentRequest) -> bool:
        raise NotImplementedError

In [7]:
class BankingSystem(ABC):
    @abstractmethod
    def process_payment(self, amount: float) -> bool:
        raise NotImplementedError


class PaytmBankingSystem(BankingSystem):
    def __init__(self) -> None:
        self._rand = random.Random()

    def process_payment(self, amount: float) -> bool:
        # Simulate 80% success (matches your Java behavior)
        r = self._rand.randrange(100)
        return r < 80


class RazorpayBankingSystem(BankingSystem):
    def __init__(self) -> None:
        self._rand = random.Random()

    def process_payment(self, amount: float) -> bool:
        print(f"[BankingSystem-Razorpay] Processing payment of {amount}...")
        # Simulate 90% success
        r = self._rand.randrange(100)
        return r < 90

class PaymentGateway(ABC):
    def __init__(self) -> None:
        self.banking_system: BankingSystem | None = None

    # Template method defining the standard payment flow
    def process_payment(self, request: PaymentRequest) -> bool:
        if not self.validate_payment(request):
            print(f"[PaymentGateway] Validation failed for {request.sender}.")
            return False
        if not self.initiate_payment(request):
            print(f"[PaymentGateway] Initiation failed for {request.sender}.")
            return False
        if not self.confirm_payment(request):
            print(f"[PaymentGateway] Confirmation failed for {request.sender}.")
            return False
        return True

    @abstractmethod
    def validate_payment(self, request: PaymentRequest) -> bool:
        raise NotImplementedError

    @abstractmethod
    def initiate_payment(self, request: PaymentRequest) -> bool:
        raise NotImplementedError

    @abstractmethod
    def confirm_payment(self, request: PaymentRequest) -> bool:
        raise NotImplementedError

In [9]:
class PaytmGateway(PaymentGateway):
    def __init__(self) -> None:
        super().__init__()
        self.banking_system = PaytmBankingSystem()

    def validate_payment(self, request: PaymentRequest) -> bool:
        print(f"[Paytm] Validating payment for {request.sender}.")
        return request.amount > 0 and request.currency == "INR"

    def initiate_payment(self, request: PaymentRequest) -> bool:
        print(f"[Paytm] Initiating payment of {request.amount} {request.currency} for {request.sender}.")
        assert self.banking_system is not None
        return self.banking_system.process_payment(request.amount)

    def confirm_payment(self, request: PaymentRequest) -> bool:
        print(f"[Paytm] Confirming payment for {request.sender}.")
        return True

In [11]:
class RazorpayGateway(PaymentGateway):
    def __init__(self) -> None:
        super().__init__()
        self.banking_system = RazorpayBankingSystem()

    def validate_payment(self, request: PaymentRequest) -> bool:
        print(f"[Razorpay] Validating payment for {request.sender}.")
        return request.amount > 0

    def initiate_payment(self, request: PaymentRequest) -> bool:
        print(f"[Razorpay] Initiating payment of {request.amount} {request.currency} for {request.sender}.")
        assert self.banking_system is not None
        return self.banking_system.process_payment(request.amount)

    def confirm_payment(self, request: PaymentRequest) -> bool:
        print(f"[Razorpay] Confirming payment for {request.sender}.")
        return True

In [17]:
class PaymentGatewayProxy(PaymentGateway):
    def __init__(self, gateway: PaymentGateway, max_retries: int) -> None:
        super().__init__()
        self._real_gateway = gateway
        self._retries = max_retries

    def process_payment(self, request: PaymentRequest) -> bool:
        result = False
        for attempt in range(self._retries):
            if attempt > 0:
                print(f"[Proxy] Retrying payment (attempt {attempt + 1}) for {request.sender}.")
            result = self._real_gateway.process_payment(request)
            if result:
                break

        if not result:
            print(f"[Proxy] Payment failed after {self._retries} attempts for {request.sender}.")
        return result

    # Delegate the steps (not strictly needed since we override process_payment,
    # but kept to match the Java structure)
    def validate_payment(self, request: PaymentRequest) -> bool:
        return self._real_gateway.validate_payment(request)

    def initiate_payment(self, request: PaymentRequest) -> bool:
        return self._real_gateway.initiate_payment(request)

    def confirm_payment(self, request: PaymentRequest) -> bool:
        return self._real_gateway.confirm_payment(request)


# ----------------------------
# Gateway Factory (Singleton-like)
# ----------------------------
class GatewayType(Enum):
    PAYTM = auto()
    RAZORPAY = auto()


class GatewayFactory:
    _instance = None

    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
        return cls._instance

    def get_gateway(self, gateway_type: GatewayType) -> PaymentGateway:
        if gateway_type == GatewayType.PAYTM:
            return PaymentGatewayProxy(PaytmGateway(), 3)
        else:
            return PaymentGatewayProxy(RazorpayGateway(), 1)


# ----------------------------
# Unified API service (Singleton-like)
# ----------------------------
class PaymentService:
    _instance = None

    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
            cls._instance.gateway = None
        return cls._instance

    def set_gateway(self, gateway: PaymentGateway) -> None:
        self.gateway = gateway

    def process_payment(self, request: PaymentRequest) -> bool:
        if self.gateway is None:
            print("[PaymentService] No payment gateway selected.")
            return False
        return self.gateway.process_payment(request)


# ----------------------------
# Controller (Singleton-like)
# ----------------------------
class PaymentController:
    _instance = None

    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
        return cls._instance

    def handle_payment(self, gateway_type: GatewayType, req: PaymentRequest) -> bool:
        payment_gateway = GatewayFactory().get_gateway(gateway_type)
        PaymentService().set_gateway(payment_gateway)
        return PaymentService().process_payment(req)


# ----------------------------
# Main (client)
# ----------------------------
if __name__ == "__main__":
    req1 = PaymentRequest("Liza", "John", 1000.0, "INR")

    print("Processing via Paytm")
    print("------------------------------")
    res1 = PaymentController().handle_payment(GatewayType.PAYTM, req1)
    print("Result:", "SUCCESS" if res1 else "FAIL")
    print("------------------------------\n")

    req2 = PaymentRequest("Liza", "John", 500.0, "USD")

    print("Processing via Razorpay")
    print("------------------------------")
    res2 = PaymentController().handle_payment(GatewayType.RAZORPAY, req2)
    print("Result:", "SUCCESS" if res2 else "FAIL")
    print("------------------------------")

Processing via Paytm
------------------------------
[Paytm] Validating payment for Liza.
[Paytm] Initiating payment of 1000.0 INR for Liza.
[Paytm] Confirming payment for Liza.
Result: SUCCESS
------------------------------

Processing via Razorpay
------------------------------
[Razorpay] Validating payment for Liza.
[Razorpay] Initiating payment of 500.0 USD for Liza.
[BankingSystem-Razorpay] Processing payment of 500.0...
[Razorpay] Confirming payment for Liza.
Result: SUCCESS
------------------------------
